In [1]:
import pygame
import time
import os
import tkinter as tk
from tkinter import *
import random
import glob
import pathlib
from PIL import Image, ImageTk


pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
season = "Sunny" #TODO User input? Random? Matching current season? 

current_time = time.asctime()
hour = current_time.split()[3].split(":")[0]
minute = current_time.split()[3].split(":")[1]

pygame.mixer.init(frequency=16000)
pygame.init()
path_music = os.getcwd()


In [2]:
# Create a dictionary associating seasons and sound with a white noise
white_noises = {
    # Outside noises
    "Sunny": ["Steps_grass.mp3", "Running_grass.mp3", "Silence.mp3"],
    "Rainy": ["Rain_umbrella.mp3", "Rain_rooftop.mp3", "Silence.mp3"],
    "Autumn": ["Steps_leaves.mp3", "Rain_umbrella.mp3", "Silence.mp3"],
    "Snowy": ["Steps_snow.mp3", "Steps_woods.mp3", "Silence.mp3", "Fireplace.mp3"],

    # Inside noises
    "Museum": ["Steps_marble.mp3"],
    "Office": ["Office_sounds.mp3"]
}

In [ ]:
from mutagen.mp3 import MP3

path = os.path.join(path_music, "acnr-soundtrack","Resident services", f"*.mp3")
files = glob.glob(path)
i = 0

for mp3_file in files:
   audio = MP3(mp3_file)
   sample_freq = audio.info.sample_rate
   print(f"File {i}: frequenza di campionamento: {sample_freq} Hz") 
   i = i + 1

In [ ]:
path = os.path.join(path_music, "acnr-soundtrack","Resident services", f"*.mp3")
songs_list = glob.glob(path)
songs_list


# Version 2: Tkinter 

In [ ]:
int(time.asctime().split()[3].split(":")[0])

In [ ]:
songs_list = ["12. Nook's Cranny (Small).mp3", "13. Nook's Cranny (Big).mp3"]
song = random.choice(songs_list)
song = os.path.join(path_music, "acnr-soundtrack", "Resident Services", song)
song
    

In [ ]:
def choose_music():
    global ambience_old
    global noise_channel
    
    if random.random() > 0.5: # Hourly music, outside noises
        ambient = "Outdoor"
        ambience_new = "out"
        current_time = time.asctime()
        hour = current_time.split()[3].split(":")[0]
        path = os.path.join(path_music, "acnr-soundtrack", season, f"{hour}_{season}*mp3")
        song = glob.glob(path)[0]
    
        # Environmental sounds
        # Select the "Folder" according to the season
        sounds_list = white_noises[season]
        sound_name = random.choice(sounds_list)
        sound = pygame.mixer.Sound(os.path.join(path_music, "acnr-soundtrack","White noises", sound_name))
        
    else: # Set outside noises
        ambience_new = "in"
        path = os.path.join(path_music, "acnr-soundtrack","Resident services", f"*.mp3")
        songs_list = glob.glob(path)
        song = random.choice(songs_list)

        if "Nook's Cranny" in song:
            hour = int(time.asctime().split()[3].split(":")[0])
            if hour <= 8 or hour > 19: # The Cranny is closed
                song = os.path.join(path_music, "acnr-soundtrack","Resident services", "14. Nook's Cranny (Before Closing).mp3")
            else: 
                songs_list = ["12. Nook's Cranny (Small).mp3", "13. Nook's Cranny (Big).mp3"]
                song = random.choice(songs_list)
                song = os.path.join(path_music, "acnr-soundtrack", "Resident Services", song)
        # These lines can be rewritten in a cleaner way. Set:
        # 1) The track for outside at 17 stops suddenly. Probably a track issue
        # 2) Set a different animation for each room. Pixel art? Cartoon style? Mineko style (Gouache + round lines)?
        if "Museum" in song:
            sound_name = random.choice(white_noises["Museum"])
        elif "Office" in song:
            sound_name = random.choice(white_noises["Office"])
        else:
            sound_name = "Silence.mp3"
            
        sound = pygame.mixer.Sound(os.path.join(path_music, "acnr-soundtrack","White noises", sound_name))

    global music_label
    music_label.config(text= (pathlib.Path(song).stem).replace("_", ".").split(".", 1)[1])
    music_label.pack()

    # Play the music
    pygame.mixer.music.load(song)
    pygame.mixer.music.play()
    # Play background sounds
    if ambience_new != ambience_old:
        noise_channel.fadeout(1000)
        noise_channel.play(sound, loops=-1, fade_ms=2000)    
    

In [ ]:
pygame.mixer.init(frequency=44100, size = -16, channels = 2, buffer = 1024)
pygame.init()
path_music = os.getcwd()
noise_channel = pygame.mixer.Channel(1) # Create a channel dedicated to background sounds
ambience_old = "None"
season = 'Sunny'

window = tk.Tk()
window.geometry("500x300")
window.title("Radio Tom Nook")
window.resizable(False, False)

# Add the background
bg = PhotoImage(file = "Pictures/Museum (Fossil Room).png")
label1 = Label(window, image = bg)
label1.place(x = 0, y = 0)
label1.image = bg

# Add the title track
music_label = Label(window, text = "")

# Show the time
time_frame = tk.Frame(window)
time_frame.pack(side= "bottom", fill = "x")
time_label = Label(time_frame, text ="")
hour_show = time.asctime().split()[3].split(":")[0]
minute_old = time.asctime().split()[3].split(":")[1]
time_label.config(text = f"{hour_show}:{minute_old}")
time_label.pack(side="right")
def update_time():
    hour_show = time.asctime().split()[3].split(":")[0]
    minute_new = time.asctime().split()[3].split(":")[1]

    global minute_old
    if minute_new != minute_old:
        time_label.config(text = f"{hour_show}:{minute_new}")
        minute_old =  minute_new
    
    time_label.after(1000, update_time)
update_time()

# Show the date. Can be updated only once, without checking every minute
date_frame = tk.Frame(window)
date_label = Label(date_frame, text= "")
# Day
day = time.asctime().split()[2]
# Month
month = time.asctime().split()[1]
months = {
    "Jan": [1], "Feb": [2], "Mar": [3], "Apr": [4], "May": [5], "Jun": [6], 
    "Jul": [7], "Aug":[8], "Sep": [9], "Oct":[10], "Nov":[11], "Dec":[12]
}
date_label.config(text= f"{day}/{months[month][0]}")
date_frame.pack(side="bottom", fill="x")
date_label.pack(side="right")

# Choose season. Order: Sunny, Rainy, Snowy
def change_season():
    global season
    if season == "Sunny":
        season = "Rainy"
        season_button['text'] = ("Rainy")
    elif season == "Rainy":
        season_button['text'] = ("Snowy")
        season = "Snowy"
    elif season == "Snowy":
        season_button['text'] = ("Sunny")
        season = "Sunny"

season_button = tk.Button(text = "Sunny", command = change_season)
season_button.place(relx = 0.8, rely = 0.1)

choose_music()
pygame.mixer.music.pause()
noise_channel.pause()

is_paused = True

def handle_music():
    if pygame.mixer.music.get_busy():
        pygame.mixer.music.pause()
        noise_channel.pause()
        pause_button['text'] = ("Play!")

    else:
        pygame.mixer.music.unpause()
        noise_channel.unpause()
        pause_button['text'] = ("Pause!")

pause_button = tk.Button(text="Play!", command = handle_music)
pause_button.place(relx = 0.5, rely = 0.85)

# Set volume: music
pygame.mixer.music.set_volume(0.2)
up_music_button = tk.Button(text = "Up", command = lambda: pygame.mixer.music.set_volume(pygame.mixer.music.get_volume() + 0.1))
down_music_button = tk.Button(text = "Down", command = lambda: pygame.mixer.music.set_volume(pygame.mixer.music.get_volume() - 0.1))
up_music_button.place(relx = 0.1, rely = 0.1)
down_music_button.place(relx = 0.2, rely = 0.1)

# Volume buttons: Background sounds
noise_channel.set_volume(0.1)
up_sounds_button = tk.Button(text = "Up", command = lambda: noise_channel.set_volume(noise_channel.get_volume() + 0.1))
down_sounds_button = tk.Button(text = "Down", command = lambda: noise_channel.set_volume(noise_channel.get_volume() - 0.1))
up_sounds_button.place(relx = 0.1, rely = 0.2)
down_sounds_button.place(relx = 0.2, rely = 0.2)

def check_music_event():
    for event in pygame.event.get():
        if event.type == MUSIC_ENDED:
            window.after(30000, choose_music)
    
    window.after(1000, check_music_event)
MUSIC_ENDED = pygame.USEREVENT + 1
pygame.mixer.music.set_endevent(MUSIC_ENDED)
check_music_event()

def on_closing():
    pygame.mixer.music.stop()
    pygame.mixer.quit()      
    window.destroy()
window.protocol("WM_DELETE_WINDOW", on_closing)

# Execute the window
if __name__ == "__main__":
    window.mainloop()


# Second version with Canvas

In [3]:
window = tk.Tk()
window.geometry("500x300")
canvas = tk.Canvas(window, width= 500, height= 300, highlightthickness=0)
canvas.pack(fill="both", expand=True)

song_name = "Museum (Fossil Room)"
if os.path.isfile(os.path.join(os.getcwd(),"Pictures", f"{song_name}.png")):
    song_path = f"Pictures/{song_name}.png"
    song_original = Image.open(song_path)
    print("The picture has been opened!")
    song_original = song_original.resize((500, 300))
    print("The picture has been resized")
    tk_song_img = ImageTk.PhotoImage(song_original)
    print("Picture has been converted to an image object")
    canvas.song_image_ref = tk_song_img


if __name__ == "__main__":
    window.mainloop()



The picture has been opened!
The picture has been resized
Picture has been converted to an image object


In [15]:
def choose_music():
    global ambience_old
    global noise_channel
    global canvas
    
    if random.random() > 1: # Hourly music, outside noises
        ambient = "Outdoor"
        ambience_new = "out"
        current_time = time.asctime()
        hour = current_time.split()[3].split(":")[0]
        path = os.path.join(path_music, "acnr-soundtrack", season, f"{hour}_{season}*mp3")
        song = glob.glob(path)[0]
    
        # Environmental sounds
        # Select the "Folder" according to the season
        sounds_list = white_noises[season]
        sound_name = random.choice(sounds_list)
        sound = pygame.mixer.Sound(os.path.join(path_music, "acnr-soundtrack","White noises", sound_name))
        
    else: # Set outside noises
        ambience_new = "in"
        path = os.path.join(path_music, "acnr-soundtrack","Resident services", f"*.mp3")
        songs_list = glob.glob(path)
        song = random.choice(songs_list)

        if "Nook's Cranny" in song:
            hour = int(time.asctime().split()[3].split(":")[0])
            if hour <= 8 or hour > 19: # The Cranny is closed
                song = os.path.join(path_music, "acnr-soundtrack","Resident services", "14. Nook's Cranny (Before Closing).mp3")
            else: 
                songs_list = ["12. Nook's Cranny (Small).mp3", "13. Nook's Cranny (Big).mp3"]
                song = random.choice(songs_list)
                song = os.path.join(path_music, "acnr-soundtrack", "Resident Services", song)
        # These lines can be rewritten in a cleaner way. Set:
        # 1) The track for outside at 17 stops suddenly. Probably a track issue
        # 2) Set a different animation for each room. Pixel art? Cartoon style? Mineko style (Gouache + round lines)?
        if "Museum" in song:
            sound_name = random.choice(white_noises["Museum"])
        elif "Office" in song:
            sound_name = random.choice(white_noises["Office"])
        else:
            sound_name = "Silence.mp3"
            
        sound = pygame.mixer.Sound(os.path.join(path_music, "acnr-soundtrack","White noises", sound_name))

    global music_label
    canvas.itemconfig(music_label, text = (pathlib.Path(song).stem).replace("_", ".").split(".", 1)[1])

    # Select picture according to the name of the song
    global sfondo_canvas
    song_name = (pathlib.Path(song).stem).replace("_", ".").split(".", 1)[1].strip()
    if os.path.isfile(os.path.join(os.getcwd(),"Pictures", f"{song_name}.png")):
        song_path = f"Pictures/{song_name}.png"
        song_pic = Image.open(song_path)
        song_pic = song_pic.resize((500, 300))
        tk_song_img = ImageTk.PhotoImage(song_pic)
        
        canvas.bg_image_ref = tk_song_img
        canvas.itemconfig(sfondo_canvas, image = tk_song_img)
        
    # Play the music
    pygame.mixer.music.load(song)
    pygame.mixer.music.play()
    # Play background sounds
    if ambience_new != ambience_old:
        noise_channel.fadeout(1000)
        noise_channel.play(sound, loops=-1, fade_ms=2000)    
    

In [37]:
pygame.mixer.init(frequency=44100, size = -16, channels = 2, buffer = 1024)
pygame.init()
path_music = os.getcwd()
noise_channel = pygame.mixer.Channel(1) # Create a channel dedicated to background sounds
ambience_old = "None"
season = 'Sunny'

window = tk.Tk()
window.geometry("500x300")
window.title("Radio Tom Nook")
window.resizable(False, False)

# Create a big canvas to place all elements
width_window = 500
height_window = 300
canvas = tk.Canvas(window, width= width_window, height= height_window, highlightthickness=0)
canvas.pack(fill="both", expand=True)

# Add the background
bg_original = Image.open("Pictures/Sunny.jpg")
bg_res = bg_original.resize((500, 300))
bg_img = ImageTk.PhotoImage(bg_res)

sfondo_canvas = canvas.create_image(0, 0, image=bg_img, anchor="nw")
canvas.bg_image_ref = bg_img

# Add the sign with time and date
width_sign = 150
height_sign = 100
Sign_original = Image.open("Pictures/Date_time_sign.png")
Sign_res = Sign_original.resize((width_sign, height_sign))
Sign_img = ImageTk.PhotoImage(Sign_res)

X_SIGN = 340
Y_SIGN = 190

canvas.create_image(X_SIGN, Y_SIGN, image=Sign_img, anchor="nw")
canvas.sign_image_ref = Sign_img

minute_old = time.asctime().split()[3].split(":")[1]
hour_show = time.asctime().split()[3].split(":")[0]
day = time.asctime().split()[2]
month = time.asctime().split()[1]
months = {
    "Jan": [1], "Feb": [2], "Mar": [3], "Apr": [4], "May": [5], "Jun": [6], 
    "Jul": [7], "Aug":[8], "Sep": [9], "Oct":[10], "Nov":[11], "Dec":[12]
}

sign_date = canvas.create_text(
    X_SIGN + (width_sign // 2), 
    Y_SIGN + 60, 
    font=("Helvetica", 13, "bold"), 
    text=f"{day}/{months[month][0]}",  
    fill="#8B0000"
)

sign_day = canvas.create_text(
    X_SIGN + (width_sign // 2) + 44,
    Y_SIGN + 60,
    font=("Helvetica", 10, "bold"), 
    text=f"{time.asctime().split()[0]}",  
    fill="white"
)

sign_time = canvas.create_text(
    X_SIGN + (width_sign // 2), 
    Y_SIGN + 80, 
    text=f"{hour_show}:{minute_old}", 
    font=("Helvetica", 13, "bold"), 
    fill="#8B0000"
)

# Add the title track
music_label = canvas.create_text(
    width_window // 2,
    11,
    text = f"{season}",
    font=("Helvetica", 13, "bold"),
    fill = "#8B0000"
)

# Update time
def update_time():
    hour_show = time.asctime().split()[3].split(":")[0]
    minute_new = time.asctime().split()[3].split(":")[1]

    global minute_old
    if minute_new != minute_old:
        canvas.itemconfig(sign_time, text = f"{hour_show}:{minute_new}")
        minute_old =  minute_new

    canvas.after(1000, update_time)
update_time()


# Choose season. Order: Sunny, Rainy, Snowy
def change_season():
    global season
    if season == "Sunny":
        season = "Rainy"
        season_button['text'] = ("Rainy")
    elif season == "Rainy":
        season_button['text'] = ("Snowy")
        season = "Snowy"
    elif season == "Snowy":
        season_button['text'] = ("Sunny")
        season = "Sunny"

season_button = tk.Button(text = "Sunny", command = change_season)
season_button.place(relx = 0.1, rely = 0.85)

choose_music()
pygame.mixer.music.pause()
noise_channel.pause()

is_paused = True

def handle_music():
    if pygame.mixer.music.get_busy():
        pygame.mixer.music.pause()
        noise_channel.pause()
        pause_button['text'] = ("Play!")

    else:
        pygame.mixer.music.unpause()
        noise_channel.unpause()
        pause_button['text'] = ("Pause!")

pause_button = tk.Button(text="Play!", command = handle_music)
pause_button.place(relx = 0.51, rely = 0.85)

# Set volume: music
pygame.mixer.music.set_volume(0.2)
up_music_button = tk.Button(text = "Up", command = lambda: pygame.mixer.music.set_volume(pygame.mixer.music.get_volume() + 0.1))
down_music_button = tk.Button(text = "Down", command = lambda: pygame.mixer.music.set_volume(pygame.mixer.music.get_volume() - 0.1))
up_music_button.place(relx = 0.1, rely = 0.1)
down_music_button.place(relx = 0.2, rely = 0.1)

# Volume buttons: Background sounds
noise_channel.set_volume(0.1)
up_sounds_button = tk.Button(text = "Up", command = lambda: noise_channel.set_volume(noise_channel.get_volume() + 0.1))
down_sounds_button = tk.Button(text = "Down", command = lambda: noise_channel.set_volume(noise_channel.get_volume() - 0.1))
up_sounds_button.place(relx = 0.1, rely = 0.2)
down_sounds_button.place(relx = 0.2, rely = 0.2)

def check_music_event():
    for event in pygame.event.get():
        if event.type == MUSIC_ENDED:
            window.after(30000, choose_music)
    
    window.after(1000, check_music_event)
MUSIC_ENDED = pygame.USEREVENT + 1
pygame.mixer.music.set_endevent(MUSIC_ENDED)
check_music_event()

def on_closing():
    pygame.mixer.music.stop()
    pygame.mixer.quit()      
    window.destroy()
window.protocol("WM_DELETE_WINDOW", on_closing)

# Execute the window
if __name__ == "__main__":
    window.mainloop()
